In [6]:
import torch
import gpytorch
import pandas as pd
import pickle
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from torch.utils.data import TensorDataset, DataLoader
from pyproj import Transformer
from sklearn.metrics import pairwise_distances
from scipy.interpolate import RegularGridInterpolator
from torch_geometric.data import Data




In [28]:
PURPLEAIR_API_KEY = "35F2A02C-D815-11EF-A3B4-42010A800010"

In [ ]:
import time
from datetime import datetime, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import requests

# ------------------------------------------------------------------
# CONFIG — edit these
# ------------------------------------------------------------------

# Bounding box covering all of South Korea
NWLAT, NWLNG = 38.6, 124.6   # Northwest corner (top-left)
SELAT, SELNG = 33.0, 131.0   # Southeast corner (bottom-right)

# Last 5 years (2021-01-01 to 2026-01-01)
START_DATE = datetime(2021, 1, 1)
END_DATE = datetime(2026, 1, 1)

# PurpleAir history endpoint limits how many days you can request per call
# depending on the averaging interval. 1440 (daily average) allows a large
# window, but we chunk conservatively to avoid timeouts/rate limits.
CHUNK_DAYS = 90

BASE_URL = "https://api.purpleair.com/v1"
HEADERS = {"X-API-Key": PURPLEAIR_API_KEY}


# ------------------------------------------------------------------
# PurpleAir: find sensors in the bounding box
# ------------------------------------------------------------------
def get_sensors_in_bbox():
    url = f"{BASE_URL}/sensors"
    params = {
        "fields": "sensor_index,name,latitude,longitude,altitude,location_type",
        "nwlat": NWLAT,
        "nwlng": NWLNG,
        "selat": SELAT,
        "selng": SELNG,
        "location_type": 0,          # 0 = outside sensors only (skip indoor units)
        "max_age": 315360000,        # Active in the last 10 years (10 years in seconds)
    }
    r = requests.get(url, headers=HEADERS, params=params, timeout=30)
    r.raise_for_status()
    data = r.json()
    return pd.DataFrame(data["data"], columns=data["fields"])


# ------------------------------------------------------------------
# PurpleAir: pull daily-averaged history for one sensor
# ------------------------------------------------------------------
def get_sensor_history(sensor_index, start_date, end_date):
    url = f"{BASE_URL}/sensors/{sensor_index}/history"
    fields = "pm2.5_atm,pm2.5_cf_1,humidity,temperature,pressure"
    rows = []

    cur = start_date
    while cur < end_date:
        nxt = min(cur + timedelta(days=CHUNK_DAYS), end_date)
        params = {
            "start_timestamp": int(cur.timestamp()),
            "end_timestamp": int(nxt.timestamp()),
            "average": 1440,  # 1440 minutes = daily average
            "fields": fields,
        }
        try:
            r = requests.get(url, headers=HEADERS, params=params, timeout=30)
            r.raise_for_status()
            data = r.json()
            if data.get("data"):
                chunk = pd.DataFrame(data["data"], columns=data["fields"])
                chunk["sensor_index"] = sensor_index
                rows.append(chunk)
        except requests.exceptions.HTTPError as e:
            print(f"  sensor {sensor_index}: HTTP error {e}")
        except requests.exceptions.RequestException as e:
            print(f"  sensor {sensor_index}: request failed {e}")
        cur = nxt
        time.sleep(1)  # stay well under PurpleAir's rate limit

    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()


def pull_all_purpleair_data():
    """Returns (sensor_metadata_df, history_df). sensor_index is kept as an
    explicit column throughout so sensor-level records can be traced back to
    a node in a graph, and sensor_metadata_df gives you the lat/lon needed to
    compute edges (e.g. by distance) between nodes."""
    sensors = get_sensors_in_bbox()
    print(f"Found {len(sensors)} sensors in bounding box")

    all_hist = []
    for _, row in sensors.iterrows():
        print(f"Pulling sensor {row['sensor_index']} ({row['name']})...")
        hist = get_sensor_history(row["sensor_index"], START_DATE, END_DATE)
        if not hist.empty:
            all_hist.append(hist)

    if not all_hist:
        return sensors, pd.DataFrame()

    df = pd.concat(all_hist, ignore_index=True)
    df["date"] = pd.to_datetime(df["time_stamp"], unit="s").dt.date

    # Basic EPA-style humidity correction for PM2.5
    df["pm25_epa_corrected"] = (
        0.52 * df["pm2.5_cf_1"] - 0.086 * df["humidity"] + 5.75
    ).clip(lower=0)

    return sensors, df


# ------------------------------------------------------------------
# Open-Meteo: historical weather for the bbox (no API key needed)
# ------------------------------------------------------------------
def get_openmeteo_weather(lat, lon, start_date, end_date, max_retries=6):
    """
    Daily temperature/humidity/precipitation/pressure/ET0 come straight from
    Open-Meteo's daily aggregates. Wind direction and speed are computed
    using vector components.
    """
    url = "https://archive-api.open-meteo.com/v1/archive"
    daily_vars = [
        "temperature_2m_max",
        "temperature_2m_min",
        "temperature_2m_mean",
        "relative_humidity_2m_mean",
        "precipitation_sum",
        "windgusts_10m_max",
        "surface_pressure_mean",
        "et0_fao_evapotranspiration",
    ]
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start_date.strftime("%Y-%m-%d"),
        "end_date": end_date.strftime("%Y-%m-%d"),
        "daily": ",".join(daily_vars),
        "hourly": "windspeed_10m,winddirection_10m",
        "timezone": "auto",
    }

    delay = 5.0
    for attempt in range(max_retries):
        r = requests.get(url, params=params, timeout=30)
        if r.status_code == 429:
            retry_after = r.headers.get("Retry-After")
            wait = float(retry_after) if retry_after else delay
            print(f"    rate limited (429) — waiting {wait:.0f}s "
                  f"(attempt {attempt + 1}/{max_retries})")
            time.sleep(wait)
            delay = min(delay * 2, 120)
            continue
        r.raise_for_status()
        data = r.json()

        daily_df = pd.DataFrame(data["daily"])
        daily_df["date"] = pd.to_datetime(daily_df["time"]).dt.date
        daily_df = daily_df.drop(columns=["time"])

        wind_df = compute_daily_wind_vector(data["hourly"])
        return daily_df.merge(wind_df, on="date", how="left")

    raise RuntimeError(f"Open-Meteo rate limit persisted after {max_retries} retries "
                        f"for ({lat}, {lon})")


def compute_daily_wind_vector(hourly):
    df = pd.DataFrame({
        "time": pd.to_datetime(hourly["time"]),
        "speed": hourly["windspeed_10m"],
        "direction": hourly["winddirection_10m"],
    })
    df["date"] = df["time"].dt.date

    rad = np.radians(df["direction"])
    df["u"] = -df["speed"] * np.sin(rad)
    df["v"] = -df["speed"] * np.cos(rad)

    daily = df.groupby("date").agg(u_mean=("u", "mean"), v_mean=("v", "mean")).reset_index()

    daily["windspeed_10m_mean"] = np.sqrt(daily["u_mean"] ** 2 + daily["v_mean"] ** 2)
    daily["winddirection_10m_prevailing"] = (
        np.degrees(np.arctan2(-daily["u_mean"], -daily["v_mean"])) % 360
    )

    return daily[["date", "windspeed_10m_mean", "winddirection_10m_prevailing"]]


_weather_cache = {}


def get_weather_for_sensor(lat, lon, start_date, end_date, round_dp=1):
    key = (round(lat, round_dp), round(lon, round_dp))
    if key not in _weather_cache:
        w = get_openmeteo_weather(key[0], key[1], start_date, end_date)
        _weather_cache[key] = w
        time.sleep(2.0)
    return _weather_cache[key]


# ------------------------------------------------------------------
# Main
# ------------------------------------------------------------------
def main():
    sensors, pa_df = pull_all_purpleair_data()
    if pa_df.empty:
        print("No PurpleAir data returned — check your API key and bbox.")
        return

    sensors.to_csv("sensor_metadata.csv", index=False)
    print(f"Saved {len(sensors)} sensors to sensor_metadata.csv")

    pa_df.to_csv("purpleair_raw.csv", index=False)
    print(f"Saved {len(pa_df)} per-sensor-day rows to purpleair_raw.csv")

    weather_out_path = "weather_by_sensor.csv"
    already_done = set()
    if Path(weather_out_path).exists():
        existing = pd.read_csv(weather_out_path)
        already_done = set(existing["sensor_index"].unique())
        print(f"Resuming: {len(already_done)} sensors already have weather saved.")

    sensor_locs = sensors[["sensor_index", "latitude", "longitude"]]
    failed_sensors = []
    for sensor_index, lat, lon in sensor_locs.itertuples(index=False):
        if sensor_index in already_done:
            continue
        print(f"Fetching weather for sensor {sensor_index} ({lat}, {lon})...")
        try:
            w = get_weather_for_sensor(lat, lon, START_DATE, END_DATE)
        except Exception as e:
            print(f"  giving up on sensor {sensor_index}: {e}")
            failed_sensors.append(sensor_index)
            continue
        w = w.copy()
        w["sensor_index"] = sensor_index
        w.to_csv(
            weather_out_path,
            mode="a",
            header=not Path(weather_out_path).exists(),
            index=False,
        )

    weather_by_sensor = pd.read_csv(weather_out_path)
    print(f"Saved {len(weather_by_sensor)} sensor-days of weather to {weather_out_path}")
    if failed_sensors:
        print(f"WARNING: {len(failed_sensors)} sensors failed weather retrieval: {failed_sensors}")

    per_sensor = pa_df.merge(sensor_locs, on="sensor_index", how="left")
    per_sensor = per_sensor.merge(
        weather_by_sensor, on=["sensor_index", "date"], how="left"
    )
    per_sensor.to_csv("purpleair_weather_by_sensor.csv", index=False)
    print(f"Saved {len(per_sensor)} rows to purpleair_weather_by_sensor.csv")


if __name__ == "__main__":
    main()

Found 192 sensors in bounding box
Pulling sensor 263837 (영남대학교 기계관 E29동)...
Pulling sensor 2394 (Yongsan-gu, Seoul)...
Pulling sensor 5302 (SCH Test2)...
Pulling sensor 5314 (dj05_OUT)...
Pulling sensor 5330 (신창면행정복지센터)...
Pulling sensor 5390 (A74D)...
Pulling sensor 5392 (sch_A4)...
Pulling sensor 5406 (sc03_04_out)...
Pulling sensor 269317 (U.S. Village)...
Pulling sensor 10046 (mob_outside)...
Pulling sensor 10060 (Air-Safety Cafe 1st_outside)...
Pulling sensor 15081 (WhiteVilla_Out)...
Pulling sensor 15087 (Air-Safety Bunker 1st_outside)...
Pulling sensor 281396 (NASA_AERONET_Hankuk_UFS)...
Pulling sensor 23369 (gcfarm)...
Pulling sensor 27899 (HeungDuk)...
Pulling sensor 28949 (Air Safety House_namyeon_outside)...
Pulling sensor 29745 (sc01_1_out)...
Pulling sensor 29747 (화덕보건진료소)...
Pulling sensor 29749 (br08_out)...
Pulling sensor 30025 (SCH Test5)...
Pulling sensor 30295 (outsidepoit)...
Pulling sensor 30309 (KH_3)...
Pulling sensor 30555 (Air Safety study Cafe_1_outside)...
Pu